# FIFA World Cup Predictor

## Version 9 - World Cup Match Predictor

### What we build

Versions 1-8 built and improved a probability model for international matches. This version wraps
that work into a **reusable World Cup prediction layer**: a function that takes any World Cup
fixture and returns match probabilities.

**Important:** we do **not** train a new model here. We reuse the exact trained model from
Version 8:

- Elo + Recent Form + difference features
- Random Forest, `n_estimators=200`, `random_state=42`
- `class_weight={0: 1.0, 1: 2.0, 2: 1.0}`
- `predict_proba()`

We also add:

- How to get each team's **latest** Elo and recent form (no future information).
- A clear **venue rule** for World Cup matches (mostly neutral, hosts only on their own soil).
- A reusable `predict_world_cup_match(team_a, team_b, neutral=True)` function.
- A clearly-marked **approximation** helper for knockout probabilities.

We do **not** build the full tournament simulator yet.

In [1]:
# Data Handling
import pandas as pd
import numpy as np

# Model Training
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import accuracy_score

# Section 1 - Load and prepare the historical data

We repeat the exact preparation from Version 8 so this notebook is self-contained. Nothing
changes about the data, features or split. We only rebuild the same inputs the model needs.

In [2]:
df = pd.read_csv("../data/raw/results.csv")

print(df.shape)
df.head()

(49477, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [3]:
df.isnull().sum()

date           0
home_team      0
away_team      0
home_score    64
away_score    64
tournament     0
city           0
country        0
neutral        0
dtype: int64

In [4]:
# Drop the 64 future World Cup fixtures that have no score yet
df = df.dropna(subset=['home_score', 'away_score']).reset_index(drop=True)

print(df.shape)

df.isnull().sum()

(49413, 9)


date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64

In [5]:
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 2      # Home Win
    elif row['home_score'] < row['away_score']:
        return 0      # Away Win
    else:
        return 1      # Draw

In [6]:
df['result'] = df.apply(get_result, axis=1)

df['result'].value_counts()

result
2    24216
0    13961
1    11236
Name: count, dtype: int64

### Elo ratings (identical to Versions 3-8)

Every team starts at 1500, ratings update after every match (K = 20), and we store each team's
rating **before** the match so there is no look-ahead.

In [7]:
teams = pd.concat([
    df['home_team'],
    df['away_team']
    ]).unique()

elo_rating = {
    team:1500
    for team in teams
    }

In [8]:
def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

def update_elo(rating, expected, actual, k=20):
    return rating + k * (actual - expected)

In [9]:
home_elos = []
away_elos = []

In [10]:
# Work through the matches in chronological order
df['date'] = pd.to_datetime(df['date'])

df = df.sort_values('date').reset_index(drop=True)

In [11]:
for _, row in df.iterrows():

    home_team = row['home_team']
    away_team = row['away_team']

    home_elo = elo_rating[home_team]
    away_elo = elo_rating[away_team]

    # Save Elo BEFORE the match
    home_elos.append(home_elo)
    away_elos.append(away_elo)

    # Expected results
    expected_home = expected_score(home_elo, away_elo)
    expected_away = expected_score(away_elo, home_elo)

    # Actual result
    if row['result'] == 2:      # Home win
        actual_home = 1
        actual_away = 0

    elif row['result'] == 0:    # Away win
        actual_home = 0
        actual_away = 1

    else:                       # Draw
        actual_home = 0.5
        actual_away = 0.5

    # Update ratings
    elo_rating[home_team] = update_elo(
        home_elo,
        expected_home,
        actual_home
    )

    elo_rating[away_team] = update_elo(
        away_elo,
        expected_away,
        actual_away
    )

In [12]:
df['home_elo'] = home_elos
df['away_elo'] = away_elos

df['elo_difference'] = df['home_elo'] - df['away_elo']

### Recent Form (identical to Versions 3-8)

Form = average points from a team's last 5 matches (Win = 1.0, Draw = 0.5, Loss = 0.0), computed
only from matches **before** the current one.

In [13]:
# How many recent matches to look back
FORM_WINDOW = 5

# Each team's recent results, stored as points (oldest first)
team_form_history = {team: [] for team in teams}

home_forms = []
away_forms = []

for _, row in df.iterrows():

    home_team = row['home_team']
    away_team = row['away_team']

    # 1) Form from matches BEFORE this one (no leakage!)
    home_history = team_form_history[home_team]
    away_history = team_form_history[away_team]

    home_form = np.mean(home_history) if len(home_history) > 0 else 0.0
    away_form = np.mean(away_history) if len(away_history) > 0 else 0.0

    # Save form BEFORE updating the histories
    home_forms.append(home_form)
    away_forms.append(away_form)

    # 2) Points for the current match
    if row['result'] == 2:      # Home win
        home_points = 1.0
        away_points = 0.0

    elif row['result'] == 0:    # Away win
        home_points = 0.0
        away_points = 1.0

    else:                       # Draw
        home_points = 0.5
        away_points = 0.5

    # 3) NOW update each team's history with the current result
    home_history.append(home_points)
    away_history.append(away_points)

    # Keep only the last 5 matches
    if len(home_history) > FORM_WINDOW:
        home_history.pop(0)
    if len(away_history) > FORM_WINDOW:
        away_history.pop(0)

In [14]:
df['home_form'] = home_forms
df['away_form'] = away_forms

### Difference features (identical to Versions 6-8)

In [15]:
df['abs_elo_difference'] = (df['home_elo'] - df['away_elo']).abs()
df['form_difference'] = df['home_form'] - df['away_form']

# Historical venue flag: 1 = neutral, 0 = home advantage
df['neutral_encoded'] = df['neutral'].astype(int)

### Chronological 80/20 split (identical to Versions 4-8)

In [16]:
split_index = int(len(df) * 0.8)

train = df.iloc[:split_index]
test = df.iloc[split_index:]

print("Training matches:    ", len(train))
print("Testing matches:     ", len(test))
print()
print("Training period:     ", train['date'].min().date(), "to", train['date'].max().date())
print("Testing period:      ", test['date'].min().date(), "to", test['date'].max().date())

Training matches:     39530
Testing matches:      9883

Training period:      1872-11-30 to 2016-03-25
Testing period:       2016-03-25 to 2026-06-13


In [17]:
feature_cols = [
    'home_elo',
    'away_elo',
    'elo_difference',
    'home_form',
    'away_form',
    'neutral_encoded',
    'abs_elo_difference',
    'form_difference'
]

X_train = train[feature_cols]
y_train = train['result']

X_test = test[feature_cols]
y_test = test['result']

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)

X_train shape: (39530, 8)
X_test shape:  (9883, 8)


# Section 2 - Recreate the Version 8 model

This is **not** a new experiment and we are **not** tuning anything. We rebuild the exact Version 8
model in this notebook so the reusable function below has access to it. The hyperparameters are
identical to Version 8.

As a sanity check we also confirm its test accuracy still matches Version 8.

In [18]:
final_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight={0: 1.0, 1: 2.0, 2: 1.0}
)
final_model.fit(X_train, y_train)

# Sanity check: same test accuracy as Version 8
test_probs = final_model.predict_proba(X_test)
test_preds = test_probs.argmax(axis=1)

print("Model trained.")
print("Test accuracy (should match Version 8's 0.5412):", round(accuracy_score(y_test, test_preds), 4))

Model trained.
Test accuracy (should match Version 8's 0.5412): 0.5412


# Section 3 - Build each team's latest Elo and form

The model needs Elo and recent form for the teams playing at the future World Cup. Those teams
have already played matches in the historical dataset - so we want their **latest available**
values.

### Only use information available before the World Cup begins

We must **never** compute Elo or form using future World Cup results - the whole point is to
predict those future matches, so the model should see nothing after today.

Our chronological processing leaves us two ready-made 'snapshots':

- `elo_rating[team]`   -> the team's rating **after its most recent historical match**.
- `team_form_history[team]` -> the team's **last 5 results** (points), as of the end of the
  historical data.

These are exactly the latest values available before the simulated tournament starts.

If a team name has no historical data at all, we fall back to a neutral default (Elo 1500,
form 0) rather than crash.

In [19]:
# Team names in the dataset - 'USA' is stored as 'United States'. Map common
# abbreviations to the exact names used in the data.
TEAM_ALIASES = {
    'USA': 'United States',
    'US': 'United States',
    'U.S': 'United States',
    'U.S.A': 'United States',
}

def canonical_team(team):
    return TEAM_ALIASES.get(team, team)

def get_latest_elo(team):
    """Latest Elo rating available for a team before the World Cup begins."""
    team = canonical_team(team)
    if team not in elo_rating:
        return 1500.0
    return elo_rating[team]

def get_latest_form(team):
    """Most recent form (average points of the last 5 matches) for a team."""
    team = canonical_team(team)
    history = team_form_history.get(team, [])
    if len(history) == 0:
        return 0.0
    return float(np.mean(history))


In [20]:
print("Latest available Elo and form for some teams:\n")
for team in ['Brazil', 'Germany', 'United States', 'Canada', 'Mexico', 'England', 'Neverland']:
    print(f"{team:15s}  Elo {get_latest_elo(team):7.0f}    form {get_latest_form(team):.3f}")


Latest available Elo and form for some teams:

Brazil           Elo    1917    form 0.700
Germany          Elo    1888    form 1.000
United States    Elo    1783    form 0.400
Canada           Elo    1756    form 0.600
Mexico           Elo    1828    form 0.900
England          Elo    1900    form 0.700
Neverland        Elo    1500    form 0.000


# Section 4 - World Cup venue logic

The 2026 World Cup is hosted by **USA, Canada and Mexico**.

### The rule

- For normal World Cup matches, treat the venue as **neutral** (`neutral_encoded = 1`)
  regardless of which team is listed first.
- A host nation only gets **home advantage** (`neutral_encoded = 0`) when that specific fixture
  is actually played on its own soil.
- We do **not** assume every match involving a host is a home match - hosts play most of their
  matches at neutral venues too.

This keeps venue information simple and explicit, so we can later feed the official 2026 schedule
(team pairings + venue per match) straight into the system.

In [21]:
# Host nations using the exact team names found in the dataset.
HOST_NATIONS = {'United States', 'Canada', 'Mexico'}

def is_host_nation(team):
    return canonical_team(team) in HOST_NATIONS

def resolve_venue(team_a, team_b, neutral=True):
    """
    Decide the neutral_encoded flag for a World Cup fixture.

    neutral=True  -> normal case: the match is at a neutral venue (neutral_encoded = 1).
    neutral=False -> the fixture is explicitly played on team_a's home soil. Only valid
                     if team_a is a host nation; otherwise we warn and fall back to
                     a neutral venue.
    """
    team_a = canonical_team(team_a)
    team_b = canonical_team(team_b)

    if neutral:
        return 1

    if is_host_nation(team_a):
        return 0                # team_a plays on its own soil -> home advantage

    print(f"Warning: '{team_a}' is not a host nation, so there is no home soil to give it. "
          f"Treating the match as neutral.")
    return 1


# Section 5 - The reusable World Cup match prediction function

Now we combine everything into one function:

```python
predict_world_cup_match(team_a, team_b, neutral=True)
```

### How it works

1. Read team A's and team B's latest Elo and form (Section 3).
2. Decide the venue flag (Section 4).
3. Feed the values into the Version 8 model with `predict_proba()`.

### Home/away input convention (important)

The model was trained on the dataset's home/away feature structure. For a **neutral** World Cup
fixture we simply use team A as the model's **"home-side input"** and team B as the
**"away-side input"**, but set `neutral_encoded = 1`.

This ordering is only an input convention for the model. It must **not** be interpreted as
real home advantage when the match is neutral.

The function returns:

- `team_a_win_probability`
- `draw_probability`
- `team_b_win_probability`
- `predicted_result`
- the venue flag used

Plus a helper to print a prediction in an easy-to-read format.

In [22]:
def predict_world_cup_match(team_a, team_b, neutral=True):
    """
    Predict the outcome probabilities for a World Cup fixture.

    Parameters
    ----------
    team_a : str   the first team (model 'home-side' input)
    team_b : str   the second team (model 'away-side' input)
    neutral : bool True for a neutral-venue match (normal case);
                   False when the fixture is explicitly played on team_a's home soil.
    """
    team_a = canonical_team(team_a)
    team_b = canonical_team(team_b)

    if team_a not in team_form_history:
        print(f"Warning: '{team_a}' has no historical data - using default Elo 1500 and form 0.")
    if team_b not in team_form_history:
        print(f"Warning: '{team_b}' has no historical data - using default Elo 1500 and form 0.")

    neutral_encoded = resolve_venue(team_a, team_b, neutral)

    home_elo = get_latest_elo(team_a)
    away_elo = get_latest_elo(team_b)
    home_form = get_latest_form(team_a)
    away_form = get_latest_form(team_b)

    # Feature order must match feature_cols exactly
    features = pd.DataFrame([{
        'home_elo': home_elo,
        'away_elo': away_elo,
        'elo_difference': home_elo - away_elo,
        'home_form': home_form,
        'away_form': away_form,
        'neutral_encoded': neutral_encoded,
        'abs_elo_difference': abs(home_elo - away_elo),
        'form_difference': home_form - away_form
    }])

    probs = final_model.predict_proba(features)[0]

    return {
        'team_a': team_a,
        'team_b': team_b,
        'team_a_win_probability': probs[2],     # class 2 = Home Win
        'draw_probability': probs[1],           # class 1 = Draw
        'team_b_win_probability': probs[0],     # class 0 = Away Win
        'predicted_result': {0: 'Team B wins', 1: 'Draw', 2: 'Team A wins'}[probs.argmax()],
        'neutral_encoded': neutral_encoded
    }

In [23]:
def show_prediction(pred):
    """Print a prediction in an easy-to-read format."""
    total = (pred['team_a_win_probability']
             + pred['draw_probability']
             + pred['team_b_win_probability'])
    venue = 'neutral venue' if pred['neutral_encoded'] == 1 else f"{pred['team_a']} home soil"
    print(f"Match: {pred['team_a']} vs {pred['team_b']}  ({venue})")
    print(f"  P({pred['team_a']} win): {pred['team_a_win_probability']:.3f}")
    print(f"  P(Draw):          {pred['draw_probability']:.3f}")
    print(f"  P({pred['team_b']} win): {pred['team_b_win_probability']:.3f}")
    print(f"  Sum: {total:.4f}    Predicted: {pred['predicted_result']}")
    print()

# Section 6 - Example predictions

Let's predict some real international fixtures.

- **Brazil vs Germany** - neutral venue (classic heavyweight match).
- **France vs Netherlands** - neutral venue.
- **USA vs Germany** - neutral venue (**no** home advantage just because USA is a host!).
- Then examples where a host is explicitly **on its own soil**:
  - **USA vs Canada** on USA soil.
  - **Mexico vs USA** on Mexico soil.
  - **Canada vs Mexico** on Canada soil.

In [24]:
examples = [
    # (team_a, team_b, neutral)
    ('Brazil', 'Germany', True),    # neutral
    ('France', 'Netherlands', True),# neutral
    ('USA', 'Germany', True),       # neutral - USA is host but no home ground here
    ('USA', 'Canada', False),       # USA home soil
    ('Mexico', 'USA', False),       # Mexico home soil
    ('Canada', 'Mexico', False),    # Canada home soil
]

all_predictions = {}

print("=== World Cup match predictions ===\n")
for team_a, team_b, neutral in examples:
    pred = predict_world_cup_match(team_a, team_b, neutral=neutral)
    all_predictions[(team_a, team_b)] = pred
    show_prediction(pred)

=== World Cup match predictions ===

Match: Brazil vs Germany  (neutral venue)
  P(Brazil win): 0.560
  P(Draw):          0.230
  P(Germany win): 0.210
  Sum: 1.0000    Predicted: Team A wins

Match: France vs Netherlands  (neutral venue)
  P(France win): 0.400
  P(Draw):          0.115
  P(Netherlands win): 0.485
  Sum: 1.0000    Predicted: Team B wins

Match: United States vs Germany  (neutral venue)
  P(United States win): 0.160
  P(Draw):          0.340
  P(Germany win): 0.500
  Sum: 1.0000    Predicted: Team B wins

Match: United States vs Canada  (United States home soil)
  P(United States win): 0.340
  P(Draw):          0.385
  P(Canada win): 0.275
  Sum: 1.0000    Predicted: Draw

Match: Mexico vs United States  (Mexico home soil)
  P(Mexico win): 0.435
  P(Draw):          0.350
  P(United States win): 0.215
  Sum: 1.0000    Predicted: Team A wins



Match: Canada vs Mexico  (Canada home soil)
  P(Canada win): 0.420
  P(Draw):          0.245
  P(Mexico win): 0.335
  Sum: 1.0000    Predicted: Team A wins



In [25]:
print("=== Verify P(A win) + P(draw) + P(B win) ~ 1 for every match ===\n")
all_ok = True
for key, pred in all_predictions.items():
    total = pred['team_a_win_probability'] + pred['draw_probability'] + pred['team_b_win_probability']
    ok = np.isclose(total, 1.0)
    all_ok = all_ok and ok
    print(f"{pred['team_a']:8s} vs {pred['team_b']:8s}   sum = {total:.6f}   {'OK' if ok else 'NOT OK'}")

print()
print("All probabilities sum to ~1:", all_ok)

=== Verify P(A win) + P(draw) + P(B win) ~ 1 for every match ===

Brazil   vs Germany    sum = 1.000000   OK
France   vs Netherlands   sum = 1.000000   OK
United States vs Germany    sum = 1.000000   OK
United States vs Canada     sum = 1.000000   OK
Mexico   vs United States   sum = 1.000000   OK
Canada   vs Mexico     sum = 1.000000   OK

All probabilities sum to ~1: True


In [26]:
summary_rows = []
for key, pred in all_predictions.items():
    summary_rows.append({
        'Match': f"{pred['team_a']} vs {pred['team_b']}",
        'Venue': 'Neutral' if pred['neutral_encoded'] == 1 else f"{pred['team_a']} home soil",
        'P(Team A)': round(pred['team_a_win_probability'], 3),
        'P(Draw)': round(pred['draw_probability'], 3),
        'P(Team B)': round(pred['team_b_win_probability'], 3),
        'Predicted': pred['predicted_result'],
    })

pd.DataFrame(summary_rows)

,Match,Venue,P(Team A),P(Draw),P(Team B),Predicted
0,Brazil vs Germany,Neutral,0.560,0.230,0.210,Team A wins
1,France vs Netherlands,Neutral,0.400,0.115,0.485,Team B wins
2,United States vs Germany,Neutral,0.160,0.340,0.500,Team B wins
3,United States vs Canada,United States home soil,0.340,0.385,0.275,Draw
4,Mexico vs United States,Mexico home soil,0.435,0.350,0.215,Team A wins
5,Canada vs Mexico,Canada home soil,0.420,0.245,0.335,Team A wins


# Section 7 - Group stage vs knockout matches

World Cup matches come in two completely different kinds, and our prediction function is built
for the group-stage kind.

### Group stage - three outcomes

Every match produces exactly one of:

- Team A wins
- Draw
- Team B wins

A draw is a perfectly valid result; both teams get a point. Our three probabilities describe
this exactly - **no conversion needed**.

### Knockout - a winner must be found

In knockout rounds a winner **must** be determined to advance:

- If the match is level after 90 minutes, play continues into **extra time** (2 x 15 minutes).
- If still level after extra time, the match goes to **penalties**.
- Either way, exactly one team advances.

So from the model's point of view the useful quantity is "which team advances", not "three
90-minute outcomes".

| | Group stage | Knockout |
|---|---|---|
| Possible outcomes | 3 (Team A / Draw / Team B) | 2 (Team A / Team B) |
| Is a draw a valid result? | Yes | No |
| 90 minutes is enough? | Yes | No - requires extra time / penalties |
| What we need to predict | win / draw / win | who advances |

# Section 8 - Preliminary knockout probability helper

We now make a **first approximation** to convert the three model probabilities into a two-team
advance probability. We deliberately keep it simple and clearly imperfect.

### The idea

Split the draw probability between the two teams **in proportion to their 90-minute win
probabilities**:

```text
P(Team A advances) = P(A wins in 90') + P(draw) * P(A wins) / (P(A wins) + P(B wins))
P(Team B advances) = P(B wins in 90') + P(draw) * P(B wins) / (P(A wins) + P(B wins))
```

### Why this is only an approximation (important)

- Extra time is not a fresh, neutral 30 minutes - tiredness, tactics and substitutions change
  the balance.
- Penalty shoot-outs are not perfectly proportional to 90-minute strength; experience and
  shot-stopping matter.
- Teams change how they play once a knockout match reaches extra time.

So treat this helper as a **starting point** to be replaced by a better extra-time / penalties
model in a later version.

In [27]:
def knockout_advance_probability(pred):
    """
    Convert three match probabilities into a two-team advance probability.

    INITIAL APPROXIMATION: the draw is split between the two teams in proportion to
    their 90-minute win probabilities. Does NOT model extra time or penalties properly.
    """
    p_a = pred['team_a_win_probability']
    p_draw = pred['draw_probability']
    p_b = pred['team_b_win_probability']

    split = p_a + p_b
    if split == 0:
        # Extremely unlikely edge case (draw probability of 1.0)
        p_a_advance = 0.5
        p_b_advance = 0.5
    else:
        p_a_advance = p_a + p_draw * (p_a / split)
        p_b_advance = p_b + p_draw * (p_b / split)

    return {
        'team_a': pred['team_a'],
        'team_b': pred['team_b'],
        'team_a_advance_probability': p_a_advance,
        'team_b_advance_probability': p_b_advance,
        'draw_after_90_minutes': p_draw
    }

In [28]:
print("=== Preliminary knockout advance probabilities ===\n")

for key in [('Brazil', 'Germany'), ('USA', 'Canada'), ('Mexico', 'USA')]:
    pred = all_predictions[key]
    knock = knockout_advance_probability(pred)
    venue = 'neutral' if pred['neutral_encoded'] == 1 else f"{pred['team_a']} home soil"
    print(f"{pred['team_a']} vs {pred['team_b']} ({venue})")
    print(f"  draw after 90': {knock['draw_after_90_minutes']:.3f}")
    print(f"  {knock['team_a']} advances: {knock['team_a_advance_probability']:.3f}")
    print(f"  {knock['team_b']} advances: {knock['team_b_advance_probability']:.3f}")
    print(f"  sum: {knock['team_a_advance_probability'] + knock['team_b_advance_probability']:.3f}")
    print()

=== Preliminary knockout advance probabilities ===

Brazil vs Germany (neutral)
  draw after 90': 0.230
  Brazil advances: 0.727
  Germany advances: 0.273
  sum: 1.000

United States vs Canada (United States home soil)
  draw after 90': 0.385
  United States advances: 0.553
  Canada advances: 0.447
  sum: 1.000

Mexico vs United States (Mexico home soil)
  draw after 90': 0.350
  Mexico advances: 0.669
  United States advances: 0.331
  sum: 1.000



# Summary

### What we built in Version 9

- A **reusable World Cup match predictor** on top of the exact Version 8 model - no new training,
  no new features, no tuning.
- Functions to read each team's latest Elo and recent form from the historical data only
  (information available before the World Cup starts).
- A clear venue rule: World Cup matches are **neutral** by default; hosts only get home
  advantage when a fixture is explicitly played **on their own soil**.
- A `predict_world_cup_match(team_a, team_b, neutral=True)` function that returns team A win,
  draw and team B win probabilities plus the predicted result.
- A **clearly-marked approximation** helper for knockout advance probabilities.

### What remains before we can simulate an entire World Cup

1. Provide the **official 2026 schedule** (fixtures + venues) so each host home game is only
   flagged when it is truly on that host's soil.
2. Define the **group composition** and the **knockout bracket**.
3. **Improve the knockout model** - properly handle extra time and penalties instead of the
   current approximation.
4. Build the **simulator loop**: play the group stage, rank each group, advance the top teams,
   then play the knockouts - sampling every match from the predicted probabilities.
5. Run many simulations and summarise each team's chances (trophy probability, final, etc.).

We deliberately stop here - the tournament simulations come in a later version.